# Compounds in varaities
This notebook builds an interactive dashboard to explore and compare compound profiles across different varieties. It loads processed data, organizes compounds by category, and provides visualizations that allow users to examine patterns, differences, and relationships between varieties. Interactive controls enable quick filtering and selection, making it easier to analyze and interpret compound distributions without rerunning computations.


In [0]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output

# Databricks / notebook
pio.renderers.default = "notebook_connected"   #"iframe"

status_df = pd.read_csv("/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/status_df.csv")
status_df["Variety"] = status_df["Variety"].astype(str).str.strip().str.upper()

try:
    results = pd.read_csv("/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/results.csv")
    results["Variety"] = results["Variety"].astype(str).str.strip().str.upper()
except:
    results = None

try:
    scores = pd.read_csv("/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/pca_scores.csv")
    scores["Variety"] = scores["Variety"].astype(str).str.strip().str.upper()
except:
    scores = None

# 
varieties = sorted(status_df["Variety"].unique().tolist())
dropdown = widgets.Dropdown(options=varieties, description="Variety:")
output = widgets.Output()

def update_dashboard(change=None):
    with output:
        clear_output(wait=True)

        target = dropdown.value
        df_var = status_df[status_df["Variety"] == target].copy()

        print(f"Selected Variety: {target}")
        print("-" * 50)

        # status counts
        counts = (
            df_var["status"]
            .value_counts()
            .reindex(["absent", "low", "present"], fill_value=0)
            .reset_index()
        )
        counts.columns = ["status", "count"]

        fig1 = px.bar(counts, x="status", y="count", title=f"Status counts for {target}")
        display(fig1)

        # top compounds
        top_df = df_var.sort_values("value", ascending=False).head(15)
        fig2 = px.bar(
            top_df.sort_values("value", ascending=True),
            x="value",
            y="compound",
            color="status",
            orientation="h",
            title=f"Top 15 compounds in {target}"
        )
        display(fig2)

        # absent / low / present
        print("\nAbsent compounds:")
        print(df_var[df_var["status"] == "absent"]["compound"].tolist())

        print("\nLow compounds:")
        print(df_var[df_var["status"] == "low"]["compound"].tolist())

        print("\nPresent compounds:")
        print(df_var[df_var["status"] == "present"]["compound"].tolist()[:20], "...")

        # outlier info
        if results is not None:
            r = results[results["Variety"] == target]
            if not r.empty:
                print("\nOutlier / Cluster info:")
                display(r)

        # PCA
        if scores is not None:
            plot_df = scores.copy()
            if results is not None:
                plot_df = plot_df.merge(results, on="Variety", how="left")

            fig3 = px.scatter(
                plot_df,
                x="PC1",
                y="PC2",
                hover_name="Variety",
                color="is_outlier" if "is_outlier" in plot_df.columns else None,
                symbol="Cluster" if "Cluster" in plot_df.columns else None,
                title="PCA Plot"
            )

            selected = plot_df[plot_df["Variety"] == target]
            if not selected.empty:
                fig3.add_scatter(
                    x=selected["PC1"],
                    y=selected["PC2"],
                    mode="markers+text",
                    text=selected["Variety"],
                    textposition="top center",
                    name="Selected Variety"
                )
            display(fig3)

display(dropdown, output)
dropdown.observe(update_dashboard, names="value")
update_dashboard()

Dropdown(description='Variety:', options=('ADORA', 'AGRIA', 'ALOUETTE', 'ALTHEA', 'ALTURAS', 'ALTUS', 'ALVERST…

Output()